# 10 — Kiểm chứng ngoài mẫu (Out-of-Sample Validation) & Phân tích chuyên môn cầu thủ

## 1. Mục tiêu
Notebook này giải thích và trực quan hóa **Bước 10** trong quy trình XAI-Football:
- Đánh giá độc lập ngoài mẫu (**Out-of-Sample**) chỉ số Hybrid Centrality ($lpha^* = 0.65$) trên tập kiểm định mù **Set B** (190 trận đấu độc lập, 314 cầu thủ ngoài khung gỗ thi đấu $\ge 90$ phút).
- So sánh đối chuẩn tương quan Spearman với 4 chỉ số bóng đá tham chiếu:
  1. **Expected Threat ($xT/90$):** Đo lường năng lực tịnh tiến bóng vào vùng đe dọa khung thành.
  2. **Expected Assists ($xA/90$):** Chất lượng cơ hội dứt điểm tạo ra từ đường chuyền (trích xuất chính thức từ `shot.statsbomb_xg`).
  3. **Key Passes ($Key Passes/90$):** Đường chuyền quyết định trực tiếp mở ra cú sút (trích xuất chính thức từ `shot.key_pass_id`).
  4. **Assists ($Assists/90$):** Đường chuyền thành bàn thắng trực tiếp.
- Đánh giá biến thể kiểm chứng đối chứng (**Ablation No-Last-Edge**): Kiểm tra sự chuyển dịch của tín hiệu giải thích khi loại bỏ đường chuyền cuối cùng.
- Kiểm định đóng góp gia tăng bằng mô hình hồi quy tuyến tính ngoài mẫu (**Out-of-Sample OLS Regression $\Delta R^2$**) với khoảng tin cậy 95% Bootstrap không kẹp.
- Thực hiện phân tích độ nhạy theo thời lượng thi đấu (90m, 450m, 900m) và xây dựng bảng xếp hạng sáng tạo cơ hội (**Creative Contribution Index**).
- Trực quan hóa phân tích trường hợp chuyên môn sân bóng 2D (**Pitch plots**) và chuyển dịch thứ hạng (**Ranking shift**).

## 2. Vị trí của bước trong pipeline
Đây là **Bước 10 — Kiểm chứng ngoài mẫu & Trực quan hóa Khóa luận (`scripts/11_run_hybrid_experiment.py`, `scripts/12_generate_all_figures.py`, `docs/thesis/ket_qua_va_thao_luan.md`)**.

## 3. Input và nguồn dữ liệu
- Điểm XAI Set B độc lập: `results/xai/hybrid/player_importance_set_b.parquet` (kèm `manifest_set_b.json`).
- Bảng kết quả chính thức:
  - `results/tables/official/hybrid_centrality_comparison.csv`
  - `results/tables/official/hybrid_centrality_baselines_comparison.csv`
  - `results/tables/official/hybrid_centrality_top_players.csv`
  - `results/tables/official/creative_contribution_top_players.csv`
  - `results/tables/official/ranking_sensitivity_by_minutes.csv`
  - `results/tables/official/hybrid_centrality_manifest.json`

## 4. Config/script/module sử dụng
- Cấu hình: `configs/hybrid_centrality.yaml`
- Scripts: `scripts/11_run_hybrid_experiment.py`, `scripts/12_generate_all_figures.py`
- Package: `src/xai_football/centrality/calculator.py`, `src/xai_football/reference/player_stats.py`

## 5. Các bước xử lý & Đọc kết quả thực tế
Đoạn mã nạp toàn bộ các bảng kết quả đối sánh ngoại mẫu và bảng xếp hạng trên Set B:

In [ ]:
import pandas as pd
import json
from pathlib import Path
from IPython.display import display, Image

tables_dir = Path("../results/tables/official")

comp_df = pd.read_csv(tables_dir / "hybrid_centrality_comparison.csv")
base_comp_df = pd.read_csv(tables_dir / "hybrid_centrality_baselines_comparison.csv")
top_df = pd.read_csv(tables_dir / "hybrid_centrality_top_players.csv")
creative_df = pd.read_csv(tables_dir / "creative_contribution_top_players.csv")
sens_df = pd.read_csv(tables_dir / "ranking_sensitivity_by_minutes.csv")

print("=== 1. ĐỐI SÁNH TƯƠNG QUAN SPEARMAN NGOẠI MẪU (SET B) ===")
display(comp_df)

print("\n=== 2. ĐỐI SÁNH CÁC BIẾN THỂ KIỂM CHỨNG ĐỐI CHỨNG (ABLATION BASELINES) ===")
display(base_comp_df)

## 6. Kết quả và cách diễn giải

### 6.1. Hồi quy tuyến tính ngoài mẫu OLS ($\Delta R^2_{	ext{OOS}}$)
Đoạn mã trích xuất kết quả kiểm định mô hình hồi quy $xT/90$ huấn luyện trên Set A và đánh giá ngoài mẫu trên Set B:

In [ ]:
with open(tables_dir / "hybrid_centrality_manifest.json", encoding="utf-8") as f:
    manifest = json.load(f)

ols_res = manifest.get("out_of_sample_ols_regression_xt", {})
print("=== KẾT QUẢ HỒI QUY OLS NGOẠI MẪU (OUT-OF-SAMPLE) CHO xT/90 ===")
print(f"R^2 (Mô hình thuần Betweenness Centrality, OOS): {ols_res.get('r2_pure_centrality'):.4f}")
print(f"R^2 (Mô hình kết hợp Centrality + XAI, OOS):      {ols_res.get('r2_full_model'):.4f}")
print(f"Delta R^2 (Mức tăng giải thích ngoài mẫu):       {ols_res.get('delta_r2'):+.4f}")
print(f"Khoảng tin cậy 95% Bootstrap không kẹp:         {ols_res.get('ci_95_bootstrap')}")
print(f"Có ý nghĩa thống kê ngoài mẫu (CI > 0):         {ols_res.get('statistically_significant')}")

### 6.2. Trực quan hóa Biểu đồ Phân tán, Dumbbell và Pitch Plots 2D

In [ ]:
fig_scatter = Path("../results/figures/official/09_hybrid_vs_centrality_scatter.png")
fig_shift = Path("../results/figures/official/10_top_players_ranking_shift.png")
fig_pitch = Path("../results/figures/official/10_case_study_pitch_plots.png")

if fig_scatter.exists():
    display(Image(filename=str(fig_scatter), width=750))
if fig_shift.exists():
    display(Image(filename=str(fig_shift), width=750))
if fig_pitch.exists():
    display(Image(filename=str(fig_pitch), width=750))

### 6.3. Bảng xếp hạng Top Cầu thủ và Tách bạch hai vai trò chiến thuật

In [ ]:
print("=== TOP 10 HYBRID CENTRALITY (NETWORK DISTRIBUTION & THREAT ENABLING) ===")
display(top_df[["player_name", "team_name", "minutes_played", "hybrid_centrality", "rank_centrality", "rank_xai", "xt_per_90", "key_passes"]].head(10))

print("\n=== TOP 10 CREATIVE CONTRIBUTION (DIRECT CHANCE CREATION / PLAYMAKING) ===")
display(creative_df[["rank", "player_name", "team_name", "creative_contribution_score", "key_passes", "key_passes_per_90", "xA_per_90", "hybrid_centrality_rank"]].head(10))

### 6.4. Phân tích chuyên môn sâu
1. **Khẳng định giá trị gia tăng thực sự ($\Delta R^2_{	ext{OOS}} = +0.0764$):**
   - Khoảng tin cậy 95% Bootstrap ngoài mẫu $[0.0120, 0.1265]$ hoàn toàn tách rời phía trên 0 mà không sử dụng bất kỳ hàm kẹp nhân tạo nào. XAI mang lại giá trị gia tăng thực tế, giải thích thêm $7.64\%$ độ biến thiên của $xT/90$.
2. **Cơ chế nhãn chia sẻ và Ablation No-Last-Edge:**
   - Điểm XAI thuần tương quan rất cao với Key Passes ($0.5889$) và $xA$ ($0.5713$) do mô hình GNN dự đoán cú sút.
   - Khi loại bỏ cạnh cuối cùng (`No Last Edge`), tương quan với Key Passes giảm về $0.1451$, nhưng tương quan với $xT/90$ lại tăng từ $0.0364$ lên $0.1337$, khẳng định XAI nắm bắt cả cấu trúc phối hợp triển khai trước đó.
3. **Tách bạch hai vai trò chiến thuật:**
   - **Nhạc trưởng toàn diện (Dual Titans):** Mesut Özil (#2 Hybrid, #1 Creative), Santi Cazorla (#1 Hybrid, #3 Creative), Cesc Fàbregas (#5 Hybrid, #10 Creative) ngự trị ở đỉnh cao của cả hai bảng xếp hạng.
   - **Chân chuyền đột biến (Final-Third Creators):** Dimitri Payet, David Silva, Dušan Tadić, Gerard Deulofeu, Christian Eriksen, Kevin De Bruyne có số Key Passes và $xA/90$ vượt trội nhưng ít tham gia phòng ngự/trục giữa nên điểm Betweenness vừa phải. Họ xếp hạng cực cao ở Creative Contribution (#1–#8) dù xếp thấp hơn ở Hybrid Centrality.
   - **Mỏ neo / Điều phối tuyến giữa (Midfield Metronomes):** Darren Fletcher, Aaron Ramsey, Yann M'Vila, Mark Noble đóng vai trò xương sống giữ nhịp kiểm soát bóng, dẫn đầu Top 10 Hybrid Centrality.

## 7. Kiểm tra chất lượng (Quality Checks)
Xác thực các tiêu chuẩn chất lượng của thực nghiệm ngoài mẫu:

In [ ]:
checks = []
checks.append({
    "Tiêu chí": "Số lượng outfield players Set B được đánh giá = 314",
    "Kết quả": "PASS" if len(top_df) == 314 else "FAIL"
})
checks.append({
    "Tiêu chí": "Delta R^2 ngoại mẫu > 0 (+0.0764)",
    "Kết quả": "PASS" if ols_res.get("delta_r2", 0) > 0 else "FAIL"
})
checks.append({
    "Tiêu chí": "Cận dưới 95% Bootstrap CI của Delta R^2 > 0 (không kẹp)",
    "Kết quả": "PASS" if ols_res.get("ci_95_bootstrap", [0, 0])[0] > 0 else "FAIL"
})
checks.append({
    "Tiêu chí": "Bảng xếp hạng đã lọc sạch thủ môn",
    "Kết quả": "PASS" if not top_df["player_name"].str.contains("Cech|De Gea|Courtois|Lloris|Hart", case=False).any() else "FAIL"
})

df_checks = pd.DataFrame(checks)
display(df_checks)
assert (df_checks["Kết quả"] == "PASS").all(), "Có kiểm tra chất lượng không đạt!"
print("Toàn bộ kiểm tra chất lượng Out-of-Sample đều ĐẠT (PASS)!")

## 8. Output được lưu ở đâu
- Bảng so sánh ngoại mẫu: `results/tables/official/hybrid_centrality_comparison.csv`
- Bảng kiểm chứng ablation đối chứng: `results/tables/official/hybrid_centrality_baselines_comparison.csv`
- Bảng xếp hạng cầu thủ: `results/tables/official/hybrid_centrality_top_players.csv`
- Bảng sáng tạo cơ hội: `results/tables/official/creative_contribution_top_players.csv`
- Bảng độ nhạy thời lượng: `results/tables/official/ranking_sensitivity_by_minutes.csv`
- Manifest: `results/tables/official/hybrid_centrality_manifest.json`
- Biểu đồ: `results/figures/official/09_hybrid_vs_centrality_scatter.png`, `10_top_players_ranking_shift.png`, `10_case_study_pitch_plots.png`

## 9. Cách tái chạy
```bash
python scripts/11_run_hybrid_experiment.py
python scripts/12_generate_all_figures.py
```

## 10. Giới hạn và giả định
- Thực nghiệm được tiến hành trên mùa giải Ngoại hạng Anh 2015/16 của StatsBomb.
- Do PGExplainer trượt Gate D17 ở cấp đồ thị possession đơn lẻ nhưng có độ ổn định cao (0.9872), chỉ số Hybrid Centrality dựa trên PGExplainer đóng vai trò là một minh chứng phương pháp luận mang tính thăm dò (exploratory methodological proof-of-concept). Khung phương pháp luận hoàn toàn sẵn sàng tích hợp các explainer mới trong tương lai.